# Musanga V3.1 · NB05 — Planet NICFI full-year export pipeline

**Overnight, restartable production notebook.** Exports every **available monthly** Planet NICFI B/G/R/N mosaic for each UAV observation year in **two distinct scopes**: (A) the UAV orthomosaic-intersected `ROI.shp` plus 250 m buffer and (B) the **full** spatial footprints from `ROI_Full.shp`, without cropping the full polygons to UAV mosaics. Per scope/site-year, it also exports annual **unscreened and conservatively screened mean + median** reflectance, raw/screened observation counts, screened-out counts, and the **annual median and temporal SD of monthly normalized NIR textures**. Full-resolution data are intended for later spatial CNN experiments.

**Important scientific constraints:** The NICFI collection begins monthly acquisitions in September 2020; Yoko 2020 is **partial-year**, and counts/filenames explicitly reflect that. The bright-observation cloud heuristic is exploratory and **not** an official cloud mask. The Baego pilot removed zero months; do not interpret 'screened' as confirmed cloud-free. Both screened and raw composites are retained. Do not substitute imagery from other years.

**Safety and restart:** All EE jobs use deterministic names; the runner inspects existing Drive outputs, persisted task IDs and current Earth Engine tasks. Existing Baego Q1/Q2/Q3/Q4 rasters and previous annual/texture files are **preserved** and reused when scope, year and bands match. No delete operation is performed. Exports are submitted in one run without waiting for completion; failed exports can be resubmitted later.

**Run:** 00 → 01 → 02 → 03 → 04 (**dry run**) → set `SUBMIT_TASKS=True` in 00 → rerun 00 and 04. Section 05 monitors tasks and outputs. Set `MAX_NEW_TASKS=None` to submit all pending exports at once (many tasks; quotas still apply).


## 00 · Settings and export policy


In [ ]:
EE_PROJECT='musanga-511110'
SOURCE_COLLECTION='projects/planet-nicfi/assets/basemaps/africa'
YEAR_BY_SITE={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
SITES=['Baego','MOS','Yangambi','Yoko']
SCOPES=['UAV_ROI','FULL_ROI']
UAV_BUFFER_M=250
FULL_BUFFER_M=0            # ROI_Full boundaries are respected; no UAV footprint clipping
SCALE_M=4.77
EXPORT_CRS='EPSG:32635'
BANDS=['B','G','R','N']
FILE_DIMENSIONS=4096
SUBMIT_TASKS=False        # First inspect Section 04 dry run, then change to True
MAX_NEW_TASKS=None       # None: submit all pending jobs in one go
RESUBMIT_FAILED=True
STRICT_FULL_ROI=True     # Stop if ROI_Full features cannot be assigned unambiguously
MIN_MONTHS_FOR_ANNUAL=2
PRINT_TASK_EVERY=10
print('EE:',EE_PROJECT,'submitting:',SUBMIT_TASKS,'cap:',MAX_NEW_TASKS)


## 01 · Colab, Drive, Earth Engine and persistent output directories


In [ ]:
from google.colab import drive
from pathlib import Path
import ee,geopandas as gpd,rasterio,pandas as pd,numpy as np
import json,datetime,hashlib,os,math,time
from shapely.geometry import box,mapping
from shapely.ops import unary_union
from IPython.display import display

if not Path('/content/drive/MyDrive').is_dir(): drive.mount('/content/drive')
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL, 'Musanga/Final missing'
DATA=FINAL/'Data/Ortho_RGB'
OUT=FINAL/'Outputs/V3_1/05_sat_predictor_exploration'
IN=OUT/'input_layers'
DER=OUT/'derived_features'
META=OUT/'metadata'
for p in [IN,DER,META]:p.mkdir(parents=True,exist_ok=True)
# The EE toDrive folder argument is a NAME, not a folder path. Every folder name below is unique.
FOLDERS={}
for scope in SCOPES:
    for family in ('Monthly','Annual','Texture'):
        dirname=f'Musanga_V31_NB05_{scope}_{family}'
        parent=DER if family=='Texture' else IN
        dest=parent/dirname
        dest.mkdir(parents=True,exist_ok=True)
        FOLDERS[(scope,family)]=(dirname,dest)
# Existing pilot cache (preserve; do not copy into a different output folder).
LEGACY_MONTHLY=IN/'Musanga_V3_1_05_Planet_NICFI_Monthly'
LEGACY_ANNUAL=IN/'Musanga_V3_1_05_Planet_Annual'
LEGACY_TEXTURE=OUT/'derived_features/Musanga_V3_1_05_Planet_Texture'
LEGACY_REGISTRY=OUT/'05_export_registry_nested_input_layers_v2.json'
assert EE_PROJECT
ee.Authenticate(auth_mode='colab')
ee.Initialize(project=EE_PROJECT)
NICFI=ee.ImageCollection(SOURCE_COLLECTION)
print('Workspace:',FINAL)
print('Earth Engine access:',NICFI.limit(1).size().getInfo(),'image test')
print('Export folders (must be uniquely named in Drive):')
for k,(name,folder) in FOLDERS.items(): print(k,folder)


## 02 · Audit the **two** ROI geometries and assign full-ROI features to forests

UAV scope: `ROI.shp` intersected with each registered RGB orthomosaic **bounding footprint**, buffered 250 m.

Full scope: `Final/Data/Ortho_RGB/ROI_Full.shp` polygons assigned **whole** to a forest by largest spatial overlap with its UAV footprint; never cropped back to UAV coverage. If any polygon intersects multiple forests or none, the audit stops rather than guessing. One large ROI_Full polygon spanning multiple forest sites must be split/attributed explicitly before exporting.


In [ ]:
MANIFEST=FINAL/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert MANIFEST.is_file(),f'Missing NB00 manifest: {MANIFEST}'
manifest=json.loads(MANIFEST.read_text())
roi_path=Path(manifest['roi'])
full_path=DATA/'ROI_Full.shp'
assert roi_path.is_file(),roi_path
assert full_path.is_file(),f'Full ROI shapefile missing: {full_path}'
roi=gpd.read_file(roi_path)
full=gpd.read_file(full_path)
assert roi.crs is not None and full.crs is not None
roi=roi.loc[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
full=full.loc[full.geometry.notna() & ~full.geometry.is_empty].copy()
assert len(roi) and len(full)
NAMES={'Baego.tif':'Baego',
       'Mos_1_2_transparent_mosaic_group1_georef.tif':'MOS',
       'Mos_5_6_transparent_mosaic_group1_georef.tif':'MOS',
       'Yangambi_georef.tif':'Yangambi',
       'Yoko_kvo_georef.tif':'Yoko'}
mosaic_parts={s:[] for s in SITES}
uav_parts={s:[] for s in SITES}
record=[]
for item in manifest['mosaics']:
    name=item['file']
    if name not in NAMES:continue
    site=NAMES[name]
    with rasterio.open(item['path']) as ds:
        assert ds.crs,name
        footprint=gpd.GeoSeries([box(*ds.bounds)],crs=ds.crs).to_crs(EXPORT_CRS).iloc[0]
        local_roi=roi.to_crs(ds.crs)
        intersection=unary_union(local_roi.geometry).intersection(box(*ds.bounds))
        mosaic_parts[site].append(footprint)
        if not intersection.is_empty:
            uav_parts[site].append(gpd.GeoSeries([intersection],crs=ds.crs).to_crs(EXPORT_CRS).iloc[0])
        record.append(dict(site=site,mosaic=name,uav_intersection_ha=round(intersection.area/1e4,2)))
display(pd.DataFrame(record))
site_fp={s:unary_union(mosaic_parts[s]) for s in SITES}
AOI_METRIC={}
for s in SITES:
    assert uav_parts[s],f'No UAV ROIs at {s}'
    AOI_METRIC[('UAV_ROI',s)]=unary_union(uav_parts[s]).buffer(UAV_BUFFER_M)
# Assign *whole* ROI_Full polygons to sites; validate no ambiguous/unmatched features.
full_metric=full.to_crs(EXPORT_CRS)
full_assign=[]
per_site={s:[] for s in SITES}
for feature_idx,geom in zip(full_metric.index,full_metric.geometry):
    intersections={s:geom.intersection(site_fp[s]).area for s in SITES}
    matches=[s for s,a in intersections.items() if a>0]
    if len(matches)==1:
        assigned=matches[0];status='assigned'
        per_site[assigned].append(geom)
    elif not matches:
        assigned=None;status='UNMATCHED'
    else:
        assigned=None;status='AMBIGUOUS_MULTI_SITE'
    full_assign.append(dict(feature_index=str(feature_idx),site=assigned,status=status,
                            polygon_ha=round(geom.area/1e4,2),
                            intersects_sites=','.join(matches)))
audit_full=pd.DataFrame(full_assign)
display(audit_full)
audit_full.to_csv(META/'02_roi_full_assignment.csv',index=False)
if STRICT_FULL_ROI:
    assert (audit_full.status=='assigned').all(), (
        'ROI_Full attribution ambiguous. Review 02_roi_full_assignment.csv and '
        'split or supply site-specific ROI_Full polygons; no silent clipping.')
for s in SITES:
    assert per_site[s],f'No ROI_Full polygon assigned to {s}'
    geom=unary_union(per_site[s])
    AOI_METRIC[('FULL_ROI',s)]=geom.buffer(FULL_BUFFER_M) if FULL_BUFFER_M else geom
AOI={}
area_rows=[]
for (scope,site),geom in AOI_METRIC.items():
    assert geom.is_valid and not geom.is_empty,(scope,site)
    geographic=gpd.GeoSeries([geom.simplify(1,preserve_topology=True)],crs=EXPORT_CRS).to_crs('EPSG:4326').iloc[0]
    AOI[(scope,site)]=ee.Geometry(mapping(geographic),geodesic=False)
    area_rows.append(dict(scope=scope,site=site,area_ha=round(geom.area/1e4,2),
        approx_4band_uncompressed_GB_per_month=round(geom.area/(SCALE_M*SCALE_M)*8/1e9,3)))
area_df=pd.DataFrame(area_rows)
display(area_df)
area_df.to_csv(META/'02_roi_scope_areas.csv',index=False)
(META/'02_roi_scope_geometries.geojson').write_text(json.dumps({
    'type':'FeatureCollection','features':[dict(type='Feature',
        properties=dict(scope=k[0],site=k[1]),geometry=mapping(
            gpd.GeoSeries([v],crs=EXPORT_CRS).to_crs('EPSG:4326').iloc[0]))
        for k,v in AOI_METRIC.items()]}))
print('ROI_Full used at full polygon extent; no clipping to the UAV mosaic.')


## 03 · Catalogue every available month in each observation year

All scopes share the same **site-year monthly NICFI image IDs**. Each month must have exactly one monthly image or it is marked missing/ambiguous. Missing Yoko January–August 2020 are not backfilled. Missing months are not silently replaced by quarterly/biannual mosaics.


In [ ]:
def month_dates(year,month):
    begin=datetime.date(year,month,1)
    end=datetime.date(year+(month==12),month%12+1,1)
    return begin.isoformat(),end.isoformat()
MONTHS={}
rows=[]
for site in SITES:
    year=YEAR_BY_SITE[site]
    for month in range(1,13):
        start,end=month_dates(year,month)
        subset=(NICFI.filterDate(start,end)
                .filter(ee.Filter.eq('cadence','monthly'))
                .filterBounds(AOI[('UAV_ROI',site)]))
        n=int(subset.size().getInfo())
        row=dict(site=site,year=year,month=month,n_images=n)
        if n==1:
            image=ee.Image(subset.first())
            info=ee.Dictionary(dict(id=image.id(),bands=image.bandNames())).getInfo()
            assert set(BANDS).issubset(set(info['bands'])),(site,month,info)
            MONTHS[(site,month)]=image.select(BANDS)
            row.update(status='available',image_id=info['id'])
        else:
            row.update(status='missing' if n==0 else 'AMBIGUOUS',image_id=None)
        rows.append(row)
catalogue=pd.DataFrame(rows)
display(catalogue.pivot(index='month',columns='site',values='status'))
catalogue.to_csv(META/'03_monthly_catalogue.csv',index=False)
assert not (catalogue.status=='AMBIGUOUS').any(),'Monthly catalogue ambiguous: stop and inspect'
for site in SITES:
    available=[m for m in range(1,13) if (site,m) in MONTHS]
    print(site,YEAR_BY_SITE[site],f'{len(available)} months:',available)
    assert len(available)>=MIN_MONTHS_FOR_ANNUAL


## 04 · **One-click batch submission** — monthly rasters + annual mean/median + NIR texture

One cell submits **all pending** jobs for all four sites × two ROI scopes. Save registry to Drive **immediately after each submitted job**. Checks actual cached files, old Baego pilot files, registry and live EE task descriptions to avoid duplicates. Skips complete exports, but a completed task with no visible files is flagged for inspection rather than blindly duplicated. EE may queue many jobs and enforce user/project quota; re-run to pick up failed jobs.

**Filename convention**: `NB05_{scope}_{site}_{year}_M{MM}_DN4.tif`, `NB05_{scope}_{site}_{year}_annual_...tif`. A legacy Baego quarterly file can satisfy a UAV-scope monthly job when its provenance matches. The historical Baego annual/texture are reused for UAV_ROI only (they have different band definitions, and are registered as legacy rather than mislabelled as the new products).


In [ ]:
# 04 — Definitions and restartable task registry. Run with SUBMIT_TASKS=False to audit.
REGISTRY_FILE=META/'04_overnight_export_registry.json'
registry=json.loads(REGISTRY_FILE.read_text()) if REGISTRY_FILE.is_file() else {}
def save_registry():
    tmp=REGISTRY_FILE.with_suffix('.json.tmp')
    tmp.write_text(json.dumps(registry,indent=2,sort_keys=True))
    os.replace(tmp,REGISTRY_FILE)
def states():
    out={}
    try:
        for task in ee.batch.Task.list():
            stat=task.status()
            if stat.get('description'):out[stat['description']]=dict(id=task.id,state=stat.get('state','UNKNOWN'))
    except Exception as exc:print('EE task list unavailable:',repr(exc))
    return out
LIVE=states()
def exists_output(scope,site,year,kind,month=None):
    foldername,dest=FOLDERS[(scope,kind)]
    if kind=='Monthly':
        prefix=f'NB05_{scope}_{site}_{year}_M{month:02d}_DN4'
    elif kind=='Annual':
        prefix=f'NB05_{scope}_{site}_{year}_Annual_19band'
    else:
        prefix=f'NB05_{scope}_{site}_{year}_Texture_9band'
    paths=sorted(dest.glob(prefix+'*.tif'))
    legacy=[]
    if scope=='UAV_ROI' and site=='Baego' and year==2023 and kind=='Monthly':
        if month in (3,6,9,12):
            q={3:1,6:2,9:3,12:4}[month]
            legacy=sorted(LEGACY_MONTHLY.glob(f'NB05_Planet_Baego_2023_Q{q}_M{month:02d}_DN4*.tif'))
        else:
            legacy=sorted(LEGACY_MONTHLY.glob(f'NB05_Planet_Baego_2023_M{month:02d}_DN4*.tif'))
    # Historical annual/texture products have different band layouts and must NOT
    # masquerade as the new 14-band annual / 9-band scoped products.
    return prefix,paths,legacy

def month_collection(site,region):
    ims=[MONTHS[(site,m)] for m in range(1,13) if (site,m) in MONTHS]
    return ee.ImageCollection.fromImages(ims).map(
        lambda im: im.updateMask(im.reduce(ee.Reducer.min()).gt(0)).clip(region))
def process_annual(site,scope):
    region=AOI[(scope,site)]
    raw=month_collection(site,region)
    reference=raw.median()
    # Haze/cloud heuristic, not UDM2. Keep screened + unscreened annual products.
    def screen(im):
        vis=im.select(['B','G','R']).reduce(ee.Reducer.mean()).multiply(0.0001)
        base=reference.select(['B','G','R']).reduce(ee.Reducer.mean()).multiply(0.0001)
        ndvi=im.normalizedDifference(['N','R'])
        bndvi=reference.normalizedDifference(['N','R'])
        bad=vis.subtract(base).gt(0.08).And(bndvi.subtract(ndvi).gt(0.10))
        return im.updateMask(bad.Not())
    screened=raw.map(screen)
    # 16 bands: 4 screened mean, 4 screened median, 4 raw mean,
    # 4 raw median, followed by 3 observation counts = 19 bands.
    means=screened.mean().rename([x+'_screen_mean' for x in BANDS])
    medians=screened.median().rename([x+'_screen_median' for x in BANDS])
    raw_mean=raw.mean().rename([x+'_raw_mean' for x in BANDS])
    raw_median=raw.median().rename([x+'_raw_median' for x in BANDS])
    nr=raw.select('N').count().rename('months_raw_valid')
    ns=screened.select('N').count().rename('months_screened_valid')
    nm=nr.subtract(ns).rename('months_screened_out')
    annual=ee.Image.cat([means,medians,raw_mean,raw_median,nr,ns,nm]).toFloat().clip(region)
    def texture(im):
        nir=im.select('N').multiply(0.0001)
        layers=[]
        for radius in (1,3,7):
            kernel=ee.Kernel.square(radius=radius,units='pixels',normalize=False)
            avg=nir.reduceNeighborhood(ee.Reducer.mean(),kernel)
            sd=nir.reduceNeighborhood(ee.Reducer.stdDev(),kernel)
            layers.append(sd.divide(avg.abs().max(ee.Image.constant(0.02))).rename(f'NIR_CV_r{radius}'))
        mean5=nir.focal_mean(radius=2,units='pixels')
        high=nir.subtract(mean5).abs().divide(mean5.abs().max(ee.Image.constant(0.02)))
        return ee.Image.cat(layers+[high.rename('NIR_abs_highpass_r2')]).toFloat()
    tcol=screened.map(texture)
    tm=tcol.median()
    names=['NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_abs_highpass_r2']
    ts=tcol.reduce(ee.Reducer.stdDev()).rename([x+'_temporal_sd' for x in names])
    nt=tcol.select('NIR_CV_r3').count().rename('texture_valid_month_count')
    tex=ee.Image.cat([tm,ts,nt]).toFloat().clip(region)
    return annual,tex

def export_one(key,scope,site,year,kind,image,month=None):
    global n_submitted
    prefix,parts,legacy=exists_output(scope,site,year,kind,month)
    desc=f'MUSANGA_V31_NB05_{scope}_{site}_{year}_'+(f'M{month:02d}' if month else kind.upper())
    rec=registry.get(key,{})
    previous=LIVE.get(desc,{})
    if parts or legacy:
        if key not in registry:
            registry[key]=dict(status='cached',paths=[str(p) for p in (parts or legacy)],
                               prefix=prefix,description=desc)
            save_registry()
        print('CACHED',key,'pieces',len(parts or legacy))
        return
    state=previous.get('state')
    if not state and rec.get('task_id'):
        try:
            st=ee.data.getTaskStatus(rec['task_id'])
            state=st[0].get('state','UNKNOWN') if st else None
        except Exception:state=None
    if state in ('READY','RUNNING','COMPLETED'):
        print('TASK',key,state,'(completed but not visible: inspect Drive folder)' if state=='COMPLETED' else '')
        return
    if rec.get('task_id') and state is None and not RESUBMIT_FAILED:
        print('UNKNOWN prior job:',key);return
    if not SUBMIT_TASKS:
        print('PENDING',key,'prior state',state or 'none');return
    if MAX_NEW_TASKS is not None and n_submitted>=MAX_NEW_TASKS:
        print('DEFERRED cap',key);return
    foldername,folder=FOLDERS[(scope,kind)]
    task=ee.batch.Export.image.toDrive(
        image=image,description=desc,folder=foldername,fileNamePrefix=prefix,
        region=AOI[(scope,site)],crs=EXPORT_CRS,scale=SCALE_M,
        fileDimensions=FILE_DIMENSIONS,shardSize=256,skipEmptyTiles=True,
        maxPixels=1e13,fileFormat='GeoTIFF',formatOptions={'cloudOptimized':True})
    task.start()
    registry[key]=dict(status='submitted',task_id=task.id,description=desc,
        scope=scope,site=site,year=year,kind=kind,month=month,
        folder=foldername,prefix=prefix,crs=EXPORT_CRS,scale_m=SCALE_M,
        band_count=4 if kind=='Monthly' else (19 if kind=='Annual' else 9))
    save_registry()
    LIVE[desc]=dict(id=task.id,state='READY')
    n_submitted+=1
    print('SUBMITTED',n_submitted,key,task.id,flush=True)
    if PRINT_TASK_EVERY and n_submitted%PRINT_TASK_EVERY==0:
        print('Checkpoint registry:',REGISTRY_FILE,flush=True)

n_submitted=0
planned=0
for site in SITES:
    year=YEAR_BY_SITE[site]
    for scope in SCOPES:
        for month in range(1,13):
            if (site,month) not in MONTHS:
                continue
            planned+=1
            key=f'{scope}/{site}/{year}/monthly/{month:02d}'
            prefix,files,legacy=exists_output(scope,site,year,'Monthly',month)
            # Avoid building large EE imagery for already cached months.
            img=MONTHS[(site,month)].toUint16().clip(AOI[(scope,site)])
            export_one(key,scope,site,year,'Monthly',img,month)
        a,t=process_annual(site,scope)
        planned+=2
        export_one(f'{scope}/{site}/{year}/annual',scope,site,year,'Annual',a)
        export_one(f'{scope}/{site}/{year}/texture',scope,site,year,'Texture',t)
print('Planned (available-month) products:',planned,'| New tasks submitted:',n_submitted)
print('Registry:',REGISTRY_FILE)


## 05 · Restart, check progress and verify actual files

Rerun 00–04 whenever Colab restarts. The pipeline uses the saved registry and EE live tasks to avoid duplicate submissions. **Completed EE task alone is not proof that a GeoTIFF exists in the expected Drive folder.** Check output sizes and GeoTIFF bands. Large full-ROI GeoTIFFs may consist of multiple parts. This QA checks existence and band metadata, not pixel-level science.


In [ ]:
# 05 — Task and Drive audit, safe to rerun.
if not REGISTRY_FILE.is_file():
    print('No exports yet.')
else:
    reg=json.loads(REGISTRY_FILE.read_text())
    audit=[]
    for key,item in sorted(reg.items()):
        if item.get('status')=='cached':
            paths=[Path(x) for x in item.get('paths',[])]
            state='CACHED'
        else:
            folder=FOLDERS[(item['scope'],item['kind'])][1]
            paths=sorted(folder.glob(item['prefix']+'*.tif'))
            try:
                st=ee.data.getTaskStatus(item['task_id'])
                state=st[0].get('state','UNKNOWN') if st else 'UNKNOWN'
            except Exception:
                state='UNKNOWN'
        audit.append(dict(job=key,ee_state=state,files=len([x for x in paths if x.exists()]),
            size_MB=round(sum(x.stat().st_size for x in paths if x.exists())/1e6,2)))
    a=pd.DataFrame(audit)
    display(a.groupby('ee_state',dropna=False).agg(jobs=('job','size'),files=('files','sum'),
                                                   total_MB=('size_MB','sum')).reset_index())
    display(a.loc[(a.files==0)|(a.ee_state=='FAILED')].head(100))
    a.to_csv(META/'05_drive_export_audit.csv',index=False)
    print('CSV:',META/'05_drive_export_audit.csv')


## 06 · Scientific next steps (after exports)

1. Inspect annual screened mean vs median vs unscreened median, and observation-count surfaces. Since the Baego pilot's heuristic mask excluded no months, call it **conservative screening**, not validated cloud removal.
2. Compare annual median monthly-NIR texture with monthly-image consistency, and measure cloud/registration sensitivity before training CNNs.
3. Train native 4.77 m Planet patch CNNs against UAV-derived 30/50/100 m fractions, carefully aligning CRS and grid; include site holdout when moving from feasibility to inference.
4. The FULL_ROI exports are spatial-context products for later inference; their large footprints are **not** UAV-labelled training data.
